# env_functions

- Este notebook armazena os métodos usados ao longo de todo o pipeline MUNIC:
  - `upsert_delta_table` — gravação/upsert de tabelas Delta particionadas por `ano_edicao`;
  - `detect_munic_sheets` / `detect_identity_renames` — detecção automática de abas e colunas de identidade (multi-edition); os overrides de exceção ficam no `env_variables`;
  - `gravar_linha_manifest` / `contrato_da_edicao` / `validar_igual` / `resumo_validacoes` — o ledger do pipeline (`pipeline_manifest`, docs/arquitetura): cada etapa declara o **contrato** do que produziu (uma linha por etapa × `ano_edicao`) e as etapas posteriores validam contra ele na fronteira.

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

## `upsert_delta_table` — como funciona

Função única de gravação do pipeline: todo notebook usa esta função para **criar ou atualizar** tabelas Delta, sempre particionadas por `ano_edicao` — é o que permite edições diferentes (2019, 2024...) conviverem na mesma tabela sem se apagarem.

**Assinatura:** `upsert_delta_table(df_source, table_name, merge_keys, ano_edicao, description)`

| Parâmetro | O que é |
| --- | --- |
| `df_source` | DataFrame Spark a ser gravado |
| `table_name` | destino completo (`catalog.schema.tabela`) |
| `merge_keys` | chaves do MERGE, ex. `["cod_munic", "ano_edicao"]` |
| `ano_edicao` | ano da edição corrente (vira coluna e partição) |
| `description` | texto gravado no histórico Delta (`DESCRIBE HISTORY` → `userMetadata`) |

**A função executa 6 passos, sempre nesta ordem — cada passo é uma função auxiliar com o mesmo nome, logo abaixo da principal:**

1. **Completar a origem** (`_adicionar_ano_edicao_na_origem`) — se o DataFrame ainda não traz `ano_edicao`, adiciona (com o ano da chamada) e o posiciona como 2ª coluna, logo após a chave da entidade.
2. **Tabela nova?** (`_criar_tabela_particionada`) — se o destino não existe no catálogo, cria já particionada por `ano_edicao` e **encerra aqui** (não há nada a atualizar).
3. **Tabela existente no formato multi-edition?** — uma de duas coisas:
   * **legado sem `ano_edicao`** (`_migrar_para_particionada`) — regrava a tabela particionada: a partição nasce de `reference_year` (cargas antigas que já tinham essa coluna) ou do parâmetro `ano_edicao` (tabelas sem ano por linha, como o dicionário);
   * **já particionada** (`_reordenar_para_padrao`) — apenas garante a ordem visual `chave da entidade, ano_edicao, resto` (não reescreve nada se já estiver assim).
4. **Evolução de schema** (`_adicionar_colunas_novas`) — colunas que existem na origem e não no destino entram via `ALTER TABLE ADD COLUMN` (ex.: edição nova traz pergunta nova, como `mtic018`).
5. **Alinhamento reverso** (`_preencher_colunas_do_destino_com_null`) — colunas que existem só no destino (herdadas de outras edições) recebem `null` na origem: o MERGE exige que toda coluna do destino exista na origem.
6. **MERGE** (`_executar_merge`) — atualiza as linhas que casam pelas `merge_keys` e insere as demais.

**Por que as `merge_keys` sempre incluem `ano_edicao`** (ex.: `["cod_munic", "ano_edicao"]`): o MERGE compara linha a linha por essas chaves, então a edição de 2024 só atualiza linhas de 2024 — nunca sobrescreve as de 2019.

**Auditoria:** toda escrita (criação, migração e MERGE) registra o texto da operação no log Delta — visível em `DESCRIBE HISTORY <tabela>`, coluna `userMetadata`.

In [0]:
def upsert_delta_table(df_source, table_name, merge_keys, ano_edicao, description):
    """Upsert de um Spark DataFrame em uma tabela Delta particionada por ano_edicao.

    Executa 6 passos, sempre nesta ordem (cada um é uma função auxiliar logo abaixo,
    com o mesmo nome — o markdown acima detalha o que cada passo faz):
      1. _adicionar_ano_edicao_na_origem   — completa a origem com a coluna de partição
      2. _criar_tabela_particionada        — tabela nova: cria e encerra
      3. _migrar_para_particionada OU
         _reordenar_para_padrao            — tabela existente no formato multi-edition
      4. _adicionar_colunas_novas          — evolução de schema (ALTER TABLE ADD)
      5. _preencher_colunas_do_destino_com_null — alinhamento reverso (null na origem)
      6. _executar_merge                    — MERGE: atualiza o que casou, insere o resto
                                            (userMetadata via conf de sessão — seção 6 do
                                             desenho do pipeline_manifest)

    Parameters
    ----------
    df_source    : Spark DataFrame a ser gravado (upsert)
    table_name   : nome totalmente qualificado da tabela de destino
    merge_keys   : lista de colunas para a condição do MERGE
    ano_edicao   : ano da edição da pesquisa (int)
    description  : string gravada no userMetadata da operação Delta
    """
    # ---- PASSO 1: a origem precisa trazer a coluna de partição 'ano_edicao' ----
    df_source, source_cols = _adicionar_ano_edicao_na_origem(df_source, ano_edicao)

    # ---- PASSO 2: tabela nova? cria particionada e encerra por aqui ----
    if not spark.catalog.tableExists(table_name):
        _criar_tabela_particionada(df_source, table_name, ano_edicao, description)
        return
    deltaTable = DeltaTable.forName(spark, table_name)

    # ---- PASSO 3: tabela existente -> garante o formato multi-edition ----
    if "ano_edicao" not in deltaTable.toDF().columns:
        # 3a. legado (criada antes da partição por edição): deriva a partição e regrava
        deltaTable, target_cols = _migrar_para_particionada(deltaTable, table_name, ano_edicao)
    else:
        # 3b. já particionada: só garante a ordem de colunas do projeto
        deltaTable, target_cols = _reordenar_para_padrao(deltaTable, table_name, merge_keys)

    # ---- PASSO 4: evolução de schema (colunas novas na origem) ----
    if _adicionar_colunas_novas(table_name, source_cols, target_cols):
        deltaTable = DeltaTable.forName(spark, table_name)  # recarrega após o ALTER

    # ---- PASSO 5: alinhamento reverso (colunas só no destino viram null na origem) ----
    df_source = _preencher_colunas_do_destino_com_null(df_source, deltaTable, source_cols, table_name)

    # ---- PASSO 6: MERGE (atualizar o que casou, inserir o que é novo) ----
    _executar_merge(deltaTable, df_source, merge_keys, table_name, ano_edicao, description)


# =============================================================================
# PASSO 1 — completar a origem com a coluna de partição
# =============================================================================
def _adicionar_ano_edicao_na_origem(df_source, ano_edicao):
    """Se 'ano_edicao' não existir na origem, adiciona e a posiciona como 2ª coluna
    (logo após a chave da entidade — ordem visual padrão do projeto).

    Retorna (df, colunas): 'colunas' reflete a ordem final da origem e é
    reaproveitada nos passos 4 e 5.
    """
    if "ano_edicao" in df_source.columns:
        return df_source, list(df_source.columns)

    primeira, restantes = df_source.columns[0], df_source.columns[1:]
    df_source = (
        df_source
        .withColumns({"ano_edicao": F.lit(ano_edicao)})
        .select(primeira, "ano_edicao", *restantes)
    )
    return df_source, [primeira, "ano_edicao", *restantes]


# =============================================================================
# PASSO 2 — criação da tabela (primeira gravação)
# =============================================================================
def _criar_tabela_particionada(df_source, table_name, ano_edicao, description):
    """Cria a tabela já no formato multi-edition: particionada por 'ano_edicao',
    com 'description' registrada no histórico Delta (userMetadata)."""
    (df_source.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .partitionBy("ano_edicao")
        .option("userMetadata", description)
        .saveAsTable(table_name))
    print(f"Criada: {table_name} - ano {ano_edicao}")


# =============================================================================
# PASSO 3 — tabela existente no formato multi-edition
# =============================================================================
def _migrar_para_particionada(deltaTable, table_name, ano_edicao):
    """Tabela legado sem 'ano_edicao': deriva a partição e regrava a tabela.

    - Se a tabela já tem 'reference_year' (cargas antigas), a partição nasce dele;
    - senão usa o 'ano_edicao' passado (ex.: tabela de dicionário, sem ano por linha).
    Regrava com 'ano_edicao' como 2ª coluna — resultado idêntico a uma tabela nova.

    Retorna o DeltaTable recarregado e as colunas resultantes.
    """
    print(f"Migrando {table_name}: adicionando a coluna de partição ano_edicao...")
    # Materializa o DataFrame uma vez para evitar RPCs de Analyze repetidos no Spark Connect
    target_df = deltaTable.toDF()
    target_cols = target_df.columns

    if "reference_year" in target_cols:
        valor_particao = F.col("reference_year").cast("int")   # partição nasce da coluna existente
    else:
        valor_particao = F.lit(ano_edicao)                     # sem reference_year -> usa o parâmetro

    primeira, restantes = target_cols[0], target_cols[1:]
    existing_df = target_df.withColumns({"ano_edicao": valor_particao}).select(
        primeira, "ano_edicao", *restantes
    )

    (existing_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .partitionBy("ano_edicao")
        .option("userMetadata", "Migracao: coluna de particao ano_edicao adicionada")
        .saveAsTable(table_name))

    return DeltaTable.forName(spark, table_name), [primeira, "ano_edicao", *restantes]


def _reordenar_para_padrao(deltaTable, table_name, merge_keys):
    """Garante a ordem de colunas do projeto: chave da entidade em 1º, 'ano_edicao' em 2º.

    Se já estiver nessa ordem, não faz nada (evita ALTER desnecessário).
    """
    target_cols = deltaTable.toDF().columns
    entity_key = next(k for k in merge_keys if k != "ano_edicao")

    ja_ordenado = (
        len(target_cols) >= 2
        and target_cols[0] == entity_key
        and target_cols[1] == "ano_edicao"
    )
    if ja_ordenado:
        return deltaTable, target_cols

    # ALTER FIRST duas vezes, nessa ordem: o resultado é [entity_key, ano_edicao, resto]
    spark.sql(f"ALTER TABLE {table_name} ALTER COLUMN ano_edicao FIRST")
    spark.sql(f"ALTER TABLE {table_name} ALTER COLUMN {entity_key} FIRST")
    target_cols = [entity_key, "ano_edicao"] + [
        c for c in target_cols if c not in (entity_key, "ano_edicao")
    ]
    return DeltaTable.forName(spark, table_name), target_cols


# =============================================================================
# PASSO 4 — evolução de schema (colunas novas na origem)
# =============================================================================
def _adicionar_colunas_novas(table_name, source_cols, target_cols):
    """Colunas que existem na origem mas não no destino: ALTER TABLE ADD COLUMN.

    Retorna True se alguma coluna foi adicionada (sinal para recarregar o DeltaTable).
    """
    faltantes = [c for c in source_cols if c not in target_cols]
    if not faltantes:
        return False
    for col in faltantes:
        spark.sql(f"ALTER TABLE {table_name} ADD COLUMN {col} STRING")
    print(f"Colunas adicionadas a {table_name}: {faltantes}")
    return True


# =============================================================================
# PASSO 5 — alinhamento reverso (colunas só no destino)
# =============================================================================
def _preencher_colunas_do_destino_com_null(df_source, deltaTable, source_cols, table_name):
    """Colunas que existem no destino mas não na origem: recebem null na origem.

    Necessário porque whenMatchedUpdateAll/whenNotMatchedInsertAll exigem que TODAS
    as colunas do destino sejam resolvíveis na origem — em upserts entre edições da
    pesquisa, o destino carrega colunas de outras edições (ex.: mtic018 existe em
    2024, mas não em 2019).
    """
    target_schema = {f.name: f.dataType for f in deltaTable.toDF().schema.fields}
    so_no_destino = [c for c in target_schema if c not in source_cols]
    if not so_no_destino:
        return df_source

    df_source = df_source.withColumns({
        c: F.lit(None).cast(target_schema[c].simpleString()) for c in so_no_destino
    })
    print(f"Colunas do destino ausentes na origem, preenchidas com null em {table_name}:\n {so_no_destino}\n")
    return df_source


# =============================================================================
# PASSO 6 — MERGE (atualizar o que casou, inserir o que é novo)
# =============================================================================
_USER_METADATA_OK = None  # None = ainda não testado; True/False após a 1a tentativa


def _set_user_metadata(description):
    """userMetadata do MERGE: só chega ao histórico via conf de sessão — MAS em
    compute serverless essa conf é CONFIG_NOT_AVAILABLE (SQLSTATE 42K0I,
    verificado em 2026-11-13). Tenta setar; quando não der, avisa UMA vez e
    segue (a trilha do commit continua com operation/timestamp do histórico)."""
    global _USER_METADATA_OK
    if _USER_METADATA_OK is False:
        return False
    try:
        spark.conf.set("spark.databricks.delta.commitInfo.userMetadata", description)
        _USER_METADATA_OK = True
        return True
    except Exception:
        _USER_METADATA_OK = False
        print("AVISO: userMetadata indisponível neste compute (serverless) — os MERGEs "
              "seguem sem a descrição no DESCRIBE HISTORY.")
        return False


def _executar_merge(deltaTable, df_source, merge_keys, table_name, ano_edicao, description=""):
    """MERGE pelas merge_keys: 'target.<k> = source.<k> AND ...'.

    Ex.: ["cod_munic", "ano_edicao"] -> a edição de 2024 atualiza apenas as
    linhas de 2024, nunca as de 2019 — é isso que mantém as edições independentes.

    userMetadata: o MERGE pela API do Delta ignora a option do write — a
    descrição só chega ao histórico (DESCRIBE HISTORY) via conf de sessão
    (docs/arquitetura/pipeline_manifest.md, seção 6). Em compute SERVERLESS
    essa conf não existe (CONFIG_NOT_AVAILABLE, verificado em 2026-11-13):
    o _set_user_metadata tenta, avisa uma vez e segue sem interromper.
    """
    if description:
        _set_user_metadata(description)
    merge_condition = " AND ".join(f"target.{k} = source.{k}" for k in merge_keys)

    (deltaTable.alias("target")
        .merge(df_source.alias("source"), merge_condition)
        .whenMatchedUpdateAll()      # linha que casou -> atualiza
        .whenNotMatchedInsertAll()   # linha que não casou -> insere
        .execute())
    print(f"MERGE concluído: {table_name} - ano {ano_edicao}")




## Detecção automática de abas e colunas (multi-edição)

**O problema.** Os nomes das abas do XLSX e das colunas que identificam o município mudam entre as edições da MUNIC — às vezes por acento, às vezes até por typo do próprio IBGE:

| O que muda | Edição 2024 | Edição 2019 |
| --- | --- | --- |
| Aba de informática | `Informática e comunicação` | `Comunicção e informática` (typo) |
| Aba de governança | `Governanca` (sem cedilha) | `Governança` |
| Código do município | `Cod Munic` | `CodMun` |
| Nome do município | `Desc Mun` | `NOME MUNIC` |
| Porte populacional | `Faixa_populacao` | `CLASSE POP` |

Antes, cada diferença dessas virava uma linha em mapas por edição (`munic_sheet_map` / `munic_identity_rename_map`) — todo ano novo de pesquisa exigia configurar de novo. **Agora a detecção é automática e não existe mais config por edição.**

**Como funciona (as funções em `env_functions` seguem exatamente este fluxo):**

1. **Normalizar** — `_nome_canonico()` reduz qualquer nome a uma forma comparável: minúsculo, sem acentos e só letras/números. Assim `Governança`, `Governanca` e `GOVERNANÇA` viram `governanca`, e `Cod Munic` vira `codmunic`.
2. **Reconhecer** — abas são localizadas por marcadores com radicais e sinônimos (`informat`/`informac` + `comunic`, `governan`, `dicion`), tolerando renomeações como 'Tecnologia da informação' ou 'TI'; colunas de identidade são localizadas por regras simples ("começa com cod e contém mun" = código do município, e assim por diante).
3. **Falhar rápido** — se mais de uma aba ou coluna casar com o padrão, a função aborta com uma mensagem explicando o que fazer, em vez de ingerir a aba/coluna errada silenciosamente.

**Como usar (o landing_to_bronze já faz as duas chamadas):**

    abas    = detect_munic_sheets(pd.ExcelFile(munic_file).sheet_names)
    renomes = detect_identity_renames(colunas_da_edicao)

**Escape opcional.** Se uma edição futura confundir a detecção (ex.: duas abas parecidas), registre um override em `munic_sheet_overrides` / `munic_identity_rename_overrides` presentes em env_variables — caso contrário, eles ficam vazios.

*Detalhe: coluna ausente numa edição (ex.: `populacao` não existe na base 2019) não gera renome — o alinhamento de schema do upsert da bronze preenche com null.*

**Divisão de responsabilidades:** a LÓGICA de detecção (funções e marcadores) mora em `env_functions`, que o `landing_to_bronze` carrega logo após este notebook; aqui ficam apenas as variáveis de configuração — caminhos, edições e os overrides de exceção (célula abaixo).


In [0]:
# ===========================================================================
# Detecção automática de abas e colunas de identidade (multi-edition)
# ===========================================================================
# Fluxo em 3 passos (documentado no markdown acima e no env_variables):
#   1. NORMALIZAR  -> reduz qualquer nome a uma forma comparável (_nome_canonico)
#   2. RECONHECER  -> procura marcas inconfundíveis (marcadores p/ abas, regras p/ colunas)
#   3. FAIL-FAST   -> se mais de uma aba/coluna casar, aborta com instrução
#
# Com isso, uma edição nova da MUNIC roda sem NENHUMA config nova por ano.
# Os overrides por edição (escape opcional) ficam no env_variables.
# ===========================================================================
import re
import unicodedata


# --- Passo 1: a forma canônica de um nome -----------------------------------
# Toda comparação abaixo usa esta forma, imune a maiúsculas, acentos,
# espaços e separadores: 'Governança' == 'Governanca' == 'governanca'.
def _nome_canonico(nome):
    """Reduz um nome a minúsculo, sem acentos e apenas letras/números.

        'Cod Munic'                -> 'codmunic'
        'Comunicção e informática' -> 'comuniccaoeinformatica'
        'Governança'               -> 'governanca'
    """
    sem_acento = unicodedata.normalize("NFKD", str(nome))
    sem_acento = "".join(ch for ch in sem_acento if not unicodedata.combining(ch))
    return re.sub(r"[^a-z0-9]", "", sem_acento.lower())


# --- Passo 2a: como reconhecer cada ABA do XLSX ------------------------------
# Cada aba é identificada por MARCADORES alternativos compostos de radicais:
# a aba casa se TODOS os tokens de QUALQUER marcador estiverem no nome canônico.
ABAS_TOKENS = {
    # Usamos radicais + sinônimos para sobreviver a renomeações entre edições
    # (ex.: 'informática' -> 'tecnologia da informação' ou 'TI').
    # chave   marcadores alternativos (tuplas de tokens)
    "dict": (
        ("dicion",),                       # 'Dicionário', 'Dicionário de dados'...
    ),
    "info": (
        ("informat", "comunic"),           # 'Informática e comunicação' | 'Comunicção e informática' (typo 2019)
        ("informac", "comunic"),           # 'Tecnologia da informação e comunicação'
        ("tiecomun",),                     # 'TI e comunicação'
    ),
    "gov": (
        ("governan",),                     # 'Governança' | 'Governanca' (sem cedilha) | 'Governança municipal'
    ),
}

def detect_munic_sheets(sheet_names):
    """A partir dos nomes de abas do XLSX, resolve {'dict': aba, 'info': aba, 'gov': aba}.

    Uso (o landing_to_bronze já faz isso na célula de leitura das planilhas):
        abas = detect_munic_sheets(pd.ExcelFile(munic_file).sheet_names)
        # -> {'dict': 'Dicionário', 'info': 'Comunicção e informática', 'gov': 'Governança'}

    Se alguma aba não for achada com exclusividade, falha aqui (fail-fast)
    com instrução de como resolver — nunca ingere a aba errada silenciosamente.
    """
    canonicos = {aba: _nome_canonico(aba) for aba in sheet_names}
    resolvidas = {}
    for alvo, marcadores in ABAS_TOKENS.items():
        achadas = [
            aba for aba, nome in canonicos.items()
            if any(all(t in nome for t in tokens) for tokens in marcadores)
        ]
        assert len(achadas) == 1, (
            f"Não consegui resolver a aba '{alvo}' (procurei os marcadores {marcadores}): "
            f"achei {achadas} entre as abas {list(sheet_names)}. "
            f"Se a edição realmente divergir, registre um override em munic_sheet_overrides."
        )
        resolvidas[alvo] = achadas[0]
    return resolvidas


# --- Passo 2b: como reconhecer cada COLUNA de identidade ---------------------
# As colunas que identificam o município mudam de nome entre edições;
# as regras abaixo reconhecem cada uma pelo seu nome canônico.
def _eh_cod_municipio(nome):    # 'CodMun' (2019) | 'Cod Munic' (2024) | 'cod_munic' (já canônico)
    return nome.startswith("cod") and "mun" in nome

def _eh_nome_municipio(nome):   # 'NOME MUNIC' (2019) | 'Desc Mun' (2024) | 'desc_mun' (já canônico)
    return "mun" in nome and ("nome" in nome or "desc" in nome)

def _eh_faixa_populacao(nome):  # 'CLASSE POP' (2019) | 'Faixa_populacao' (2024)
    return any(pedaco in nome for pedaco in ("classe", "faixa", "porte"))

# (nome canônico na bronze, o que a coluna significa, regra de reconhecimento)
COLUNAS_IDENTIDADE = [
    ("cod_munic",       "código IBGE do município",   _eh_cod_municipio),
    ("desc_mun",        "nome do município",          _eh_nome_municipio),
    ("faixa_populacao", "faixa de porte populacional", _eh_faixa_populacao),
]

def detect_identity_renames(colunas):
    """A partir das colunas de uma edição, gera os renomes {original: canônico} para a bronze.

    Uso (o landing_to_bronze já faz isso antes do upsert):
        renomes = detect_identity_renames(['codmun', 'nome_munic', 'classe_pop', 'mtic011', ...])
        # -> {'codmun': 'cod_munic', 'nome_munic': 'desc_mun', 'classe_pop': 'faixa_populacao'}

    Observações:
    - Coluna já canônica (ex.: 2024 'Cod Munic' -> 'cod_munic') não gera renome.
    - Coluna ausente numa edição (ex.: 'populacao' não existe em 2019) também não gera renome:
      o upsert da bronze completa colunas faltantes com null (alinhamento de schema).
    """
    renomes = {}
    for alvo, _significado, reconhece in COLUNAS_IDENTIDADE:
        candidatas = [col for col in colunas if col != alvo and reconhece(_nome_canonico(col))]
        assert len(candidatas) <= 1, (
            f"A coluna canônica '{alvo}' ficou ambígua: candidatas {candidatas} entre {colunas}. "
            f"Registre um override em munic_identity_rename_overrides."
        )
        if candidatas:
            renomes[candidatas[0]] = alvo
    return renomes

In [0]:
# ===========================================================================
# Ledger do pipeline — pipeline_manifest (docs/arquitetura/pipeline_manifest.md)
# ===========================================================================
# O livro-razão do pipeline: UMA linha por etapa × ano_edicao em que cada etapa
# declara o CONTRATO do que produziu — quantas linhas, quantos indicadores com
# dados, quanto de recusa preservou, a partir de qual arquivo (SHA-256).
#
# Papéis (desenho, seção 1.1): o ledger é primariamente CONTRATO (validado nas
# fronteiras por validar_igual), secundariamente AUDITORIA (cada atualização da
# linha é uma versão no DESCRIBE HISTORY da própria tabela) e deliberadamente
# NÃO é evidência — isso continua com as dq_* do bronze_quality_profiling.
#
# O MAP 'detalhes' guarda contexto, diagnóstico e relatório: não entra nos
# asserts por padrão, mas localiza o desvio quando um check reprova — e sobe a
# coluna contratual quando passa a ser sistematicamente validado (seção 5.5).
#
# 'manifest_tbl' vem do env_variables; a tabela nasce na primeira gravação.
# ===========================================================================

import io
import contextlib

SCHEMA_MANIFEST = """
  etapa            STRING,
  ano_edicao       INT,
  tabela_destino   STRING,
  sha256_xlsx      STRING,
  linhas           BIGINT,
  indicadores      INT,
  recusa_total     BIGINT,
  detalhes         MAP<STRING, STRING>,
  criado_em        TIMESTAMP
"""


def gravar_linha_manifest(etapa, ano_edicao, tabela_destino="", sha256_xlsx=None,
                          linhas=None, indicadores=None, recusa_total=None, detalhes=None):
    """Grava/atualiza a linha etapa×ano_edicao do ledger (MERGE na chave).

    É o estado atual, não o histórico: re-rodar a etapa atualiza a linha, nunca
    duplica (o histórico completo das escritas continua no DESCRIBE HISTORY de
    cada tabela — e do próprio ledger). Chamada por cada etapa ao concluir sua
    gravação: landing (ingest), bronze, profiling, silver e gold.
    """
    if not spark.catalog.tableExists(manifest_tbl):
        spark.sql(f"CREATE TABLE IF NOT EXISTS {manifest_tbl} ({SCHEMA_MANIFEST}) USING DELTA")
        print(f"Criada: {manifest_tbl}")

    df_linha = spark.createDataFrame(
        [(etapa, int(ano_edicao), tabela_destino or None, sha256_xlsx,
          int(linhas) if linhas is not None else None,
          int(indicadores) if indicadores is not None else None,
          int(recusa_total) if recusa_total is not None else None)],
        schema="etapa string, ano_edicao int, tabela_destino string, sha256_xlsx string, "
               "linhas bigint, indicadores int, recusa_total bigint",
    )
    if detalhes:
        pares = [F.lit(x) for k, v in sorted(detalhes.items()) for x in (k, str(v))]
        mapa = F.create_map(*pares)
    else:
        mapa = F.lit(None).cast("map<string,string>")
    df_linha = (df_linha
                .withColumn("detalhes", mapa)
                .withColumn("criado_em", F.current_timestamp()))

    # MERGE pela API do Delta: userMetadata só chega ao histórico via conf de sessão
    # (indisponível em compute serverless — o _set_user_metadata avisa uma vez e segue)
    _set_user_metadata(f"Ledger: {etapa} - edicao {ano_edicao}")

    # O commit do MERGE emite ~2 linhas em branco no stdout (ruído da Delta em
    # serverless) — capturamos e reimprimimos apenas as linhas com conteúdo
    _stdout_merge = io.StringIO()
    with contextlib.redirect_stdout(_stdout_merge):
        (DeltaTable.forName(spark, manifest_tbl).alias("target")
            .merge(df_linha.alias("source"),
                   "target.etapa = source.etapa AND target.ano_edicao = source.ano_edicao")
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute())
    for _linha in _stdout_merge.getvalue().splitlines():
        if _linha.strip():
            print(_linha)

    print(f"Ledger atualizado: etapa '{etapa}', edição {ano_edicao} -> {manifest_tbl}")


def contrato_da_edicao(etapa, ano_edicao):
    """Retorna a linha do ledger (o contrato) da etapa×edição, como dict.

    Aborta com instrução clara quando a linha não existe (padrão da
    _exigir_dq_da_edicao da gold) e também quando as tabelas físicas que a
    linha declara já não confirmam o contrato (DROP manual): o ledger declara,
    a guard confere (seção 5.4 do desenho).
    """
    if not spark.catalog.tableExists(manifest_tbl):
        raise RuntimeError(
            f"Ledger ausente: {manifest_tbl} não existe. As etapas do pipeline gravam "
            f"suas linhas ao executar (comece pelo pipeline/ingest_to_landing)."
        )
    linhas = (spark.table(manifest_tbl)
              .filter((F.col("etapa") == etapa) & (F.col("ano_edicao") == int(ano_edicao)))
              .collect())
    if not linhas:
        raise RuntimeError(
            f"Contrato ausente no ledger: não há linha da etapa '{etapa}' para a edição "
            f"{ano_edicao} em {manifest_tbl}. Rode a etapa '{etapa}' desta edição primeiro "
            f"(ela declara seu contrato ao concluir a gravação)."
        )
    contrato = linhas[0].asDict()

    # A existência real é confirmada na tabela física, não no ledger (DROP manual)
    for tabela in [t.strip() for t in (contrato.get("tabela_destino") or "").split(";") if t.strip()]:
        if tabela.startswith("/"):
            continue  # caminho de volume (linha 'landing'), não é tabela — nada a conferir
        if not spark.catalog.tableExists(tabela):
            raise RuntimeError(
                f"Contrato desmentido: o ledger declara '{etapa}/{ano_edicao}', mas a tabela "
                f"{tabela} não existe (apagada manualmente?). Rode novamente a etapa "
                f"'{etapa}' da edição {ano_edicao}."
            )
        if "ano_edicao" in spark.table(tabela).columns:
            n = spark.table(tabela).filter(F.col("ano_edicao") == int(ano_edicao)).count()
            if n == 0:
                raise RuntimeError(
                    f"Contrato desmentido: {tabela} existe, mas não tem linhas da edição "
                    f"{ano_edicao}. Rode novamente a etapa '{etapa}' desta edição."
                )
    return contrato


_VALIDACOES = []  # acumula os validar_igual até o resumo_validacoes


def validar_igual(obtido, esperado, nome_check, contexto=""):
    """Confere igualdade contra o contrato; a falha NÃO interrompe o run aqui — acumula.

    Todas as reprovações interrompem juntas no resumo_validacoes (padrão do Step 9
    do silver_to_gold: checks PASS/FAIL). 'contexto' entra na mensagem de falha
    e serve para citar os valores de 'detalhes' da linha do ledger — o
    diagnóstico que localiza o desvio sem uma segunda consulta (seção 5.5).
    """
    ok = obtido == esperado
    _VALIDACOES.append({"check": nome_check, "esperado": esperado, "obtido": obtido})
    print(f"[{'PASS' if ok else 'FAIL'}] {nome_check}: esperado {esperado}, obtido {obtido}")
    if not ok and contexto:
        print(f"    contexto: {contexto}")
    return ok


def resumo_validacoes():
    """Fecha o lote de validar_igual: resume PASS/FAIL e, se algo reprovou,
    interrompe com TODAS as falhas listadas de uma vez (fail-fast em bloco)."""
    falhas = [v for v in _VALIDACOES if v["obtido"] != v["esperado"]]
    print(f"\nValidação contra o contrato: {len(_VALIDACOES) - len(falhas)}/{len(_VALIDACOES)} checks PASS")
    if falhas:
        lista = "\n".join(
            f"  - {f['check']}: esperado {f['esperado']}, obtido {f['obtido']}" for f in falhas
        )
        raise AssertionError(f"{len(falhas)} check(s) reprovado(s) contra o contrato do ledger:\n{lista}")
    _VALIDACOES.clear()

In [0]:
# ===========================================================================
# Ledger, etapa 'landing': verificação do insumo antes de declarar
# ===========================================================================
# Encapsula o que é específico da landing (o artefato é ARQUIVO, não tabela):
# SHA-256 recalculado em TODO run, aviso de divergência contra o último
# declarado (a linha nunca é atualizada em silêncio) e baixado_em honesto
# (now se este run baixou; mtime do arquivo se já existia).
# NÃO grava no ledger — retorna os valores da linha; quem declara é o notebook,
# chamando gravar_linha_manifest como toda etapa faz. Desenho: §5.6.
# ===========================================================================
import datetime
import hashlib
import os


def verificar_arquivo_landing(ano_edicao, url_origem, file_path, baixou):
    """Verifica o arquivo da edição e monta os valores da linha 'landing'.

    baixou: retorno de download_file (True se este run baixou, False se já existia).
    Retorna (sha256_xlsx, detalhes) prontos para o gravar_linha_manifest.
    """
    file_name = url_origem.split("/")[-1]
    with open(file_path, "rb") as f:
        file_bytes = f.read()
    file_sha256 = hashlib.sha256(file_bytes).hexdigest()

    print(f"Arquivo : {file_name}")
    print(f"Volume  : {file_path}")
    print(f"Tamanho : {len(file_bytes):,} bytes")
    print(f"SHA-256 : {file_sha256}\n")

    # Último contrato declarado — para avisar divergência (seção 5.6)
    sha_anterior = None
    if spark.catalog.tableExists(manifest_tbl):
        linhas_ledger = (
            spark.table(manifest_tbl)
            .filter((F.col("etapa") == "landing") & (F.col("ano_edicao") == int(ano_edicao)))
            .select("sha256_xlsx")
            .collect()
        )
        if linhas_ledger and linhas_ledger[0]["sha256_xlsx"]:
            sha_anterior = linhas_ledger[0]["sha256_xlsx"]

    # 'baixado_em': quando o arquivo chegou ao volume — agora se este run baixou;
    # mtime do arquivo se já existia (nunca um 'agora' fabricado)
    momento_download = (
        datetime.datetime.now() if baixou
        else datetime.datetime.fromtimestamp(os.path.getmtime(file_path))
    )
    detalhes = {
        "arquivo": file_name,
        "tamanho_mb": f"{len(file_bytes) / (1024 * 1024):.1f}",
        "url_origem": url_origem,
        "baixado_em": momento_download.strftime("%Y-%m-%d %H:%M"),
        "ja_existia": str(not baixou).lower(),
    }
    if sha_anterior and sha_anterior != file_sha256:
        print(f"AVISO: SHA-256 da edição {ano_edicao} divergiu do último declarado no ledger "
              f"— o insumo da landing mudou (exclusão + re-download, origem alterada "
              f"ou arquivo substituído).")
        print(f"    anterior: {sha_anterior}")
        print(f"    novo    : {file_sha256}\n")
        detalhes["sha_anterior_divergente"] = sha_anterior

    return file_sha256, detalhes